In [4]:
import re
import time
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager


# ================= CONFIG =================

SECTIONS = {
    "market-intelligence": "https://agrowon.esakal.com/market-intelligence",
    "agro-special": "https://agrowon.esakal.com/agro-special",
    "weather-news": "https://agrowon.esakal.com/weather-news"
}

BASE_URL = "https://agrowon.esakal.com"

ARTICLE_PATTERN = re.compile(
    r"https://agrowon\.esakal\.com/.+/.+-[a-z0-9]+$"
)

MAX_WORKERS = 3  # Selenium is heavy — keep low


# ================= DRIVER =================

import os

def create_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-dev-shm-usage")

    driver_path = ChromeDriverManager().install()

    # Fix for Mac webdriver-manager bug
    if "THIRD_PARTY_NOTICES" in driver_path:
        driver_path = os.path.join(os.path.dirname(driver_path), "chromedriver")

    service = Service(driver_path)

    driver = webdriver.Chrome(
        service=service,
        options=options
    )

    return driver

# ================= UTIL =================

def clean(text):
    return re.sub(r"\s+", " ", text).strip()


# ================= COLLECT URLS =================

def collect_urls():
    driver = create_driver()
    all_urls = set()

    try:
        for name, url in SECTIONS.items():
            print(f"\n🔎 Collecting URLs from: {name}")
            driver.get(url)
            time.sleep(3)

            # Scroll to load more
            for _ in range(4):
                driver.execute_script(
                    "window.scrollTo(0, document.body.scrollHeight);"
                )
                time.sleep(2)

            soup = BeautifulSoup(driver.page_source, "html.parser")

            for a in soup.find_all("a", href=True):
                href = a["href"]

                if href.startswith("/"):
                    href = BASE_URL + href

                if ARTICLE_PATTERN.match(href):
                    all_urls.add(href)

    finally:
        driver.quit()

    print(f"\n✅ Total unique article URLs collected: {len(all_urls)}")
    return list(all_urls)


# ================= SCRAPE ARTICLE =================

def scrape_article(url):

    driver = create_driver()

    try:
        driver.get(url)

        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "h1"))
        )

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # -------- TITLE --------
        title_tag = soup.find("h1")
        title = clean(title_tag.get_text()) if title_tag else ""

        # -------- PUBLISHED DATE --------
        published_date = ""
        publish_block = soup.find("div", {"data-test-id": "publishDetails"})
        if publish_block:
            time_tag = publish_block.find("time")
            if time_tag and time_tag.get("datetime"):
                published_date = time_tag["datetime"]

        # -------- UPDATED DATE --------
        updated_date = ""
        update_block = soup.find("div", {"data-test-id": "updateDetails"})
        if update_block:
            time_tag = update_block.find("time")
            if time_tag and time_tag.get("datetime"):
                updated_date = time_tag["datetime"]

        # -------- CONTENT --------
        paragraphs = []
        for p in soup.find_all("p"):
            text = clean(p.get_text())
            if len(text) > 40:
                paragraphs.append(text)

        content = "\n\n".join(paragraphs)

        return {
            "section": url.split("/")[3],
            "title": title,
            "url": url,
            "published_datetime": published_date,
            "updated_datetime": updated_date,
            "content": content,
            "content_length": len(content),
            "scrape_date": datetime.now().strftime("%Y-%m-%d")
        }

    except Exception:
        return None

    finally:
        driver.quit()


# ================= MAIN =================

def main():

    start = time.time()

    urls = collect_urls()

    print("\n⚡ Scraping articles in parallel...\n")

    articles = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(scrape_article, url) for url in urls]

        for future in as_completed(futures):
            result = future.result()
            if result and result["content_length"] > 300:
                articles.append(result)
                print("✔", result["title"][:70])

    df = pd.DataFrame(articles).drop_duplicates(subset=["url"])

    print("\n================ SUMMARY ================")
    print("Total valid articles:", len(df))
    print("⏱ Time taken:", round(time.time() - start, 2), "seconds")

    return df


if __name__ == "__main__":
    df = main()
    print("\nSample output:\n")
    print(df.head())


ConnectionError: Could not reach host. Are you offline?

In [8]:
import os
import re
import ssl
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

# ================= NETWORK & SSL FIX =================
# Forces Python and Selenium to bypass strict SSL verification environments
os.environ["WDM_SSL_VERIFY"] = "0"
ssl._create_default_https_context = ssl._create_unverified_context

# ================= CONFIG =================

SECTIONS = {
    "market-intelligence": "https://agrowon.esakal.com/market-intelligence",
    "agro-special": "https://agrowon.esakal.com/agro-special",
    "weather-news": "https://agrowon.esakal.com/weather-news",
}

BASE_URL = "https://agrowon.esakal.com"

ARTICLE_PATTERN = re.compile(r"https://agrowon\.esakal\.com/.+/.+-[a-z0-9]+$")

MAX_WORKERS = 3  # Selenium is heavy — keep low


# ================= DRIVER =================


def create_driver():
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-dev-shm-usage")

    # CLEAN & NATIVE SELENIUM 4+ SETUP:
    # No ChromeDriverManager, no paths, and no 'THIRD_PARTY_NOTICES' Mac bug fixes needed.
    driver = webdriver.Chrome(options=options)

    return driver


# ================= UTIL =================


def clean(text):
    return re.sub(r"\s+", " ", text).strip()


# ================= COLLECT URLS =================


def collect_urls():
    driver = create_driver()
    all_urls = set()

    try:
        for name, url in SECTIONS.items():
            print(f"\n🔎 Collecting URLs from: {name}")
            driver.get(url)
            time.sleep(3)

            # Scroll to load more
            for _ in range(4):
                driver.execute_script(
                    "window.scrollTo(0, document.body.scrollHeight);"
                )
                time.sleep(2)

            soup = BeautifulSoup(driver.page_source, "html.parser")

            for a in soup.find_all("a", href=True):
                href = a["href"]

                if href.startswith("/"):
                    href = BASE_URL + href

                if ARTICLE_PATTERN.match(href):
                    all_urls.add(href)

    finally:
        driver.quit()

    print(f"\n✅ Total unique article URLs collected: {len(all_urls)}")
    return list(all_urls)


# ================= SCRAPE ARTICLE =================


def scrape_article(url):

    driver = create_driver()

    try:
        driver.get(url)

        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "h1"))
        )

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # -------- TITLE --------
        title_tag = soup.find("h1")
        title = clean(title_tag.get_text()) if title_tag else ""

        # -------- PUBLISHED DATE --------
        published_date = ""
        publish_block = soup.find("div", {"data-test-id": "publishDetails"})
        if publish_block:
            time_tag = publish_block.find("time")
            if time_tag and time_tag.get("datetime"):
                published_date = time_tag["datetime"]

        # -------- UPDATED DATE --------
        updated_date = ""
        update_block = soup.find("div", {"data-test-id": "updateDetails"})
        if update_block:
            time_tag = update_block.find("time")
            if time_tag and time_tag.get("datetime"):
                updated_date = time_tag["datetime"]

        # -------- CONTENT --------
        paragraphs = []
        for p in soup.find_all("p"):
            text = clean(p.get_text())
            if len(text) > 40:
                paragraphs.append(text)

        content = "\n\n".join(paragraphs)

        return {
            "section": url.split("/")[3],
            "title": title,
            "url": url,
            "published_datetime": published_date,
            "updated_datetime": updated_date,
            "content": content,
            "content_length": len(content),
            "scrape_date": datetime.now().strftime("%Y-%m-%d"),
        }

    except Exception:
        return None

    finally:
        driver.quit()


# ================= MAIN =================


def main():

    start = time.time()

    urls = collect_urls()

    print("\n⚡ Scraping articles in parallel...\n")

    articles = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(scrape_article, url) for url in urls]

        for future in as_completed(futures):
            result = future.result()
            if result and result["content_length"] > 300:
                articles.append(result)
                print("✔", result["title"][:70])

    df = pd.DataFrame(articles).drop_duplicates(subset=["url"])

    print("\n================ SUMMARY ================")
    print("Total valid articles:", len(df))
    print("⏱ Time taken:", round(time.time() - start, 2), "seconds")

    return df


if __name__ == "__main__":
    df = main()
    print("\nSample output:\n")
    print(df.head())



🔎 Collecting URLs from: market-intelligence

🔎 Collecting URLs from: agro-special

🔎 Collecting URLs from: weather-news

✅ Total unique article URLs collected: 66

⚡ Scraping articles in parallel...

✔ Legislative Reform: पदच्युतीऐवजी निलंबनाची शिफारस
✔ Bacchu Kadu: शेतीमालाच्या हमीभावासाठी लढा उभारणारच; बच्चू कडू यांची घो
✔ Monsoon Rain: काही भागात हलक्या पावसाची शक्यता; राज्यातील बहुतांशी भाग
✔ US Iran Conflict: अमेरिकेचे इराणवर तीव्र हल्ले; इराणकडूनही आखाती देश ल
✔ Interview With Manikrao Khule: मराठवाड्यात जोरदार पाऊस कधी पडणार?
✔ Maharashtra Rain News: राज्यात पावसाची उघडीप; उन्हाचा चटका वाढला
✔ Banana Trade: तेल्हाऱ्याची केळी निघाली जम्मू-काश्मीरच्या दिशेने
✔ Crop Damage Compensation: राज्यातील २१ जिल्ह्यांतील शेतकऱ्यांना दिलासा
✔ Stamp Duty Relief: शहीद जवानांच्या वारसदारांना मुद्रांक शुल्कातून दिला
✔ Agriculture Export Growth: कृषी उत्पादनांच्या बळावर बुलडाण्याची निर्या
✔ Eknath Shinde: ते देशात अराजकता पसरवण्याचे काम करताहेत, एकनाथ शिंदेंचा
✔ Crop Insurance Model: ‘पॅरामेट्रि

In [9]:
df

,section,title,url,published_datetime,updated_datetime,content,content_length,scrape_date
0,agro-special,Legislative Reform: पदच्युतीऐवजी निलंबनाची शिफारस,https://agrowon.esakal.com/agro-special/jpc-re...,2026-07-13 07:00:00 GMT+5:30,2026-07-13 07:00:00 GMT+5:30,Indian Politics: गंभीर गुन्ह्यांमध्ये सलग ३० द...,2704,2026-07-14
1,agro-special,Bacchu Kadu: शेतीमालाच्या हमीभावासाठी लढा उभार...,https://agrowon.esakal.com/agro-special/bachch...,2026-07-13 08:15:00 GMT+5:30,2026-07-13 08:15:00 GMT+5:30,Bachchu Kadu Farmers Protest Announcement: शेत...,2128,2026-07-14
2,weather-news,Monsoon Rain: काही भागात हलक्या पावसाची शक्यता...,https://agrowon.esakal.com/weather-news/mahara...,2026-07-14 15:15:00 GMT+5:30,2026-07-14 15:15:00 GMT+5:30,Maharashtra Monsoon Update: राज्यातील बहुतांशी...,1851,2026-07-14
3,agro-special,US Iran Conflict: अमेरिकेचे इराणवर तीव्र हल्ले...,https://agrowon.esakal.com/agro-special/us-ira...,2026-07-13 06:30:00 GMT+5:30,2026-07-13 06:30:00 GMT+5:30,US Airstrikes on Iran after Hormuz Attack: होर...,3098,2026-07-14
4,agro-special,Interview With Manikrao Khule: मराठवाड्यात जोर...,https://agrowon.esakal.com/agro-special/solapu...,2026-07-12 12:30:00 GMT+5:30,2026-07-12 12:30:00 GMT+5:30,Weather Prediction: राज्यात जुलै महिन्यात पहिल...,7607,2026-07-14
5,weather-news,Maharashtra Rain News: राज्यात पावसाची उघडीप; ...,https://agrowon.esakal.com/weather-news/mahara...,2026-07-10 15:28:41 GMT+5:30,2026-07-10 15:28:41 GMT+5:30,Maharashtra Monsoon Update: कोकण आणि घाटमाथ्या...,1785,2026-07-14
6,market-intelligence,Banana Trade: तेल्हाऱ्याची केळी निघाली जम्मू-क...,https://agrowon.esakal.com/market-intelligence...,2026-07-14 07:00:00 GMT+5:30,2026-07-14 07:00:00 GMT+5:30,Banana Farming Success Story: अकोला जिल्ह्याती...,2005,2026-07-14
7,agro-special,Crop Damage Compensation: राज्यातील २१ जिल्ह्य...,https://agrowon.esakal.com/agro-special/crop-d...,2026-07-13 14:06:34 GMT+5:30,2026-07-13 14:06:34 GMT+5:30,Crop Damage Compensation GR Maharashtra: राज्य...,2189,2026-07-14
8,agro-special,Stamp Duty Relief: शहीद जवानांच्या वारसदारांना...,https://agrowon.esakal.com/agro-special/stamp-...,2026-07-12 17:30:00 GMT+5:30,2026-07-12 17:30:00 GMT+5:30,Revenue Department Policy: शहीद जवान व अधिकाऱ्...,2569,2026-07-14
9,agro-special,Agriculture Export Growth: कृषी उत्पादनांच्या ...,https://agrowon.esakal.com/agro-special/buldha...,2026-07-12 07:00:00 GMT+5:30,2026-07-12 07:00:00 GMT+5:30,"Buldhana Farmers Export Success: कृषी उत्पादन,...",2320,2026-07-14


In [6]:
df.to_csv('agrowon_news_data_10_03.csv', index=False) 

In [8]:
df.to_excel('agrowon_news_data_final.xlsx')

NameError: name 'df' is not defined